# Parallelization

This pattern enables additional efficiencies by running multiple sub-tasks in parallel and then collecting results for synthesis. Use cases include: deep research, travel planning, customer feedback analysis, etc.

## Implementation with Flyte v2

This notebook refactors the LangChain `RunnableParallel` example to use Flyte v2 and the native OpenAI client. The three parallel sub-tasks (summarize, questions, key terms) run as Flyte tasks that can execute in parallel containers when run remotely.

#### LangChain vs Flyte v2 — Key Differences

In the context of this pattern, these are the main differences between Langchain and Flyte V2:


| Aspect | LangChain | Flyte v2 |
|---------|-----------|----------|
| **Parallel execution** | `RunnableParallel` (LCEL) | `asyncio.gather` + Flyte tasks |
| **LLM calls** | `ChatOpenAI` via LCEL | Direct `AsyncOpenAI` client |
| **Caching** | None | `cache="auto"` per task |
| **Execution** | In-process only | Local or remote (containers) |
| **Dependencies** | langchain-openai, langchain-core | flyte, openai |


1. Install dependencies

In [2]:
!uv pip install 'flyte[tui]' openai

Using Python 3.12.12 environment at: /Users/davidmirror/code/agentic-patterns-on-v2/.venv
Audited 2 packages in 3ms


2. Export your API key

In [ ]:
%env OPENAI_API_KEY=sk-proj-...

3. Import dependencies and configure the Flyte TaskEnvironment

In [ ]:
import os
import asyncio
import flyte
from flyte import TaskEnvironment, Resources, Secret
from openai import AsyncOpenAI

flyte.init(
    endpoint="<your-union-endpoint-url>",  # Your Union cluster URL
    org="<your-union-org>",                                     # Your organization
    project="<your-project>",                               # Your project name
    domain="development",                           # Environment: development/staging/production
    image_builder="remote",                         # Build images on cluster (no local Docker needed)
    auth_type="DeviceFlow",
)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY not set. Use %env OPENAI_API_KEY=... and re-run this cell."
    )
client = AsyncOpenAI(api_key=api_key)


env = TaskEnvironment(
    name="parallel_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    image=flyte.Image.from_debian_base().with_pip_packages("openai"),
)

4. Define the three parallel sub-tasks and the synthesis task

Each sub-task is a Flyte task that calls OpenAI directly. When run remotely, Flyte schedules them in parallel containers.

In [ ]:
async def _llm_call(system: str, user: str) -> str:
    """Helper: single LLM call with system + user messages."""
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.7,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return (resp.choices[0].message.content or "").strip()


@env.task
async def summarize_task(topic: str) -> str:
    """Summarize the topic concisely."""
    return await _llm_call(
        "Summarize the following topic concisely:",
        topic,
    )


@env.task
async def questions_task(topic: str) -> str:
    """Generates questions about the topic."""
    return await _llm_call(
        "Generate three interesting questions about the following topic:",
        topic,
    )


@env.task
async def terms_task(topic: str) -> str:
    """Identify 5-10 key terms from the topic, comma-separated."""
    return await _llm_call(
        "Identify 5-10 key terms from the following topic, separated by commas:",
        topic,
    )


@env.task
async def synthesize_task(
    topic: str, summary: str, questions: str, key_terms: str
) -> str:
    """Synthesize a comprehensive answer from the parallel results."""
    system = """Based on the following information:
Summary: {summary}
Related Questions: {questions}
Key Terms: {key_terms}
Synthesize a comprehensive answer."""
    return await _llm_call(
        system.format(summary=summary, questions=questions, key_terms=key_terms),
        f"Original topic: {topic}",
    )

5. Orchestrate the parallel workflow

The main task runs the three sub-tasks in parallel with `asyncio.gather`. Flyte schedules them concurrently (locally or in separate containers when remote).

In [6]:
@env.task
async def parallel_research(topic: str) -> str:
    """Run summarize, questions, and terms in parallel, then synthesize."""
    summary, questions, key_terms = await asyncio.gather(
        summarize_task(topic),
        questions_task(topic),
        terms_task(topic),
    )
    return await synthesize_task(topic, summary, questions, key_terms)

6. Run locally

In [ ]:
# Local execution (same pattern as routing.ipynb)
if "OPENAI_API_KEY" not in os.environ:
    print("Set OPENAI_API_KEY in your environment before running this demo.")
else:
    run = flyte.with_runcontext(mode="local").run(
        parallel_research, topic="The history of space exploration"
    )
    run.wait()
    print(run.outputs()[0])

The history of space exploration can be traced back to early astronomical observations and theoretical concepts about the universe, but it gained significant momentum during the mid-20th century, particularly influenced by the geopolitical rivalry of the Cold War. This period marked crucial milestones that shaped the trajectory of space exploration. The launch of Sputnik 1 by the Soviet Union in 1957 is often regarded as the dawn of the space age, as it was the first artificial satellite to orbit Earth, triggering a fierce competition between the U.S. and the USSR. This rivalry led to remarkable achievements, including the first human in space, Yuri Gagarin, in 1961, and culminated in the U.S. Apollo program's historic manned moon landing in 1969, which solidified America’s position in the space race.

As the Cold War eased, space exploration diversified beyond national rivalry to include international collaboration. The development of the International Space Station (ISS) represents a

### Running remotely

1. Create the secret to be stored by the remote cluster:

In [ ]:
!flyte create secret OPENAI_API_KEY --value sk-proj-...

2. Modify your `TaskEnvironment` to include a request for secrets:

In [ ]:
env = TaskEnvironment(
    name="parallel_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    image=flyte.Image.from_debian_base().with_pip_packages("openai"),
    secrets=[Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")],
)

3. Change the run configuration to use the remote (default) mode:

In [ ]:
if __name__ == "__main__":
    flyte.init_from_config()
    run = flyte.run(parallel_research, topic="The history of space exploration")
    run.wait()
    print(run.outputs()[0])

## Scaling the pattern

a. Use `ReusePolicy` to keep a pool of warm containers instead of starting a new one per action. When you have multiple short tasks (LLM calls) where container startup time dominates overall latency, Flyte's reusable containers eliminate that factor by enabling the action to run in an already-warm container.